In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
PROJECT_ROOT = '/content/drive/MyDrive/dissertation_cryptosent'
for sub in ['data/raw/prices', 'data/raw/twitter',
            'data/processed', 'models', 'results']:
    os.makedirs(f'{PROJECT_ROOT}/{sub}', exist_ok=True)
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import requests
import pandas as pd
import numpy as np
from datetime import datetime
import time

In [ ]:
def fetch_binance_klines(symbol: str, interval: str,
                        start_date: str, end_date: str) -> pd.DataFrame:
    """
    Fetch historical OHLCV from Binance's public klines endpoint.
    Binance caps responses at 1000 candles, so we paginate by start time.
    """
    base_url = 'https://api.binance.com/api/v3/klines'
    start_ts = int(pd.Timestamp(start_date, tz='UTC').timestamp() * 1000)
    end_ts   = int(pd.Timestamp(end_date,   tz='UTC').timestamp() * 1000)

    all_klines = []
    current_start = start_ts

    while current_start < end_ts:
        params = {
            'symbol': symbol,
            'interval': interval,
            'startTime': current_start,
            'endTime': end_ts,
            'limit': 1000,
        }
        response = requests.get(base_url, params=params, timeout=30)
        response.raise_for_status()
        batch = response.json()
        if not batch:
            break
        all_klines.extend(batch)
        current_start = batch[-1][0] + 1
        time.sleep(0.25)  # be polite to the free endpoint

    df = pd.DataFrame(all_klines, columns=[
        'open_time', 'open', 'high', 'low', 'close', 'volume',
        'close_time', 'quote_volume', 'trades',
        'taker_base', 'taker_quote', 'ignore'
    ])
    df['date'] = pd.to_datetime(df['open_time'], unit='ms', utc=True).dt.date
    for col in ['open', 'high', 'low', 'close', 'volume']:
        df[col] = df[col].astype(float)
    return df[['date', 'open', 'high', 'low', 'close', 'volume']]

In [ ]:
# Provisional window — we may shift this once we choose the Twitter dataset.
# Fetching extra is cheap; we can always trim later.
btc = fetch_binance_klines('BTCUSDT', '1d', '2021-01-01', '2024-12-31')

print(f'BTC: {len(btc)} rows, {btc.date.min()} → {btc.date.max()}')
btc.head()

In [ ]:
def validate_daily_prices(df: pd.DataFrame, name: str):
    df = df.copy()
    df['date'] = pd.to_datetime(df['date'])
    dupes   = df['date'].duplicated().sum()
    full    = pd.date_range(df['date'].min(), df['date'].max(), freq='D')
    missing = set(full) - set(df['date'])
    invalid = ((df[['open','high','low','close']] <= 0).any(axis=1)).sum()
    print(f'--- {name} ---')
    print(f'  Rows            : {len(df)}')
    print(f'  Duplicate dates : {dupes}')
    print(f'  Missing dates   : {len(missing)}')
    print(f'  Invalid prices  : {invalid}')

validate_daily_prices(btc, 'BTC')

In [ ]:
btc.to_parquet(f'{PROJECT_ROOT}/data/raw/prices/btc_daily.parquet', index=False)
print('Saved BTC daily prices.')

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(12, 4))
plt.plot(pd.to_datetime(btc.date), btc.close, color='#F7931A')  # Bitcoin orange
plt.title('BTC/USDT Daily Close, 2021–2024')
plt.ylabel('USDT')
plt.xlabel('Date')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
!pip install -q kaggle kagglehub

In [ ]:
from google.colab import files
print('Upload your kaggle.json file (just downloaded from Kaggle)')
files.upload()

# Move it to the standard location
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print('Kaggle authenticated.')

In [ ]:
PROJECT_ROOT = '/content/drive/MyDrive/dissertation_cryptosent'
TWITTER_RAW  = f'{PROJECT_ROOT}/data/raw/twitter'

# Download directly into Drive (so we don't re-download on session restart)
!kaggle datasets download -d kaushiksuresh147/bitcoin-tweets -p {TWITTER_RAW} --unzip

In [ ]:
import os
files_in_dir = os.listdir(TWITTER_RAW)
print('Files in twitter raw directory:')
for f in sorted(files_in_dir):
    path = f'{TWITTER_RAW}/{f}'
    size_mb = os.path.getsize(path) / (1024**2)
    print(f'  {f:50s}  {size_mb:8.1f} MB')

In [ ]:
import pandas as pd

print('=== Bitcoin_tweets.csv (main file) ===')
df_sample_main = pd.read_csv(f'{TWITTER_RAW}/Bitcoin_tweets.csv', nrows=100)
print(f'Columns ({len(df_sample_main.columns)}):')
for col in df_sample_main.columns:
    print(f'  {col:30s}  dtype={df_sample_main[col].dtype}')
print()
print('First 3 rows:')
print(df_sample_main.head(3).to_string())

In [ ]:
print('\n=== Bitcoin_tweets_dataset_2.csv (supplementary) ===')
df_sample_2 = pd.read_csv(f'{TWITTER_RAW}/Bitcoin_tweets_dataset_2.csv', nrows=100)
print(f'Columns ({len(df_sample_2.columns)}):')
for col in df_sample_2.columns:
    print(f'  {col:30s}  dtype={df_sample_2[col].dtype}')
print()
print('First 3 rows:')
print(df_sample_2.head(3).to_string())

In [ ]:
import pandas as pd

def summarise_csv_dates(path: str, date_col: str = 'date',
                       chunksize: int = 100_000) -> dict:
    """
    Stream-read a CSV and return row count + date range.
    Memory-bounded: never holds more than `chunksize` rows at once.
    """
    total_rows = 0
    bad_dates  = 0
    min_date   = None
    max_date   = None

    for chunk in pd.read_csv(path, usecols=[date_col],
                             chunksize=chunksize, low_memory=False):
        # Parse dates; invalid strings become NaT (Not a Time)
        parsed = pd.to_datetime(chunk[date_col], errors='coerce', utc=True)
        bad_dates  += parsed.isna().sum()
        valid       = parsed.dropna()
        total_rows += len(chunk)

        if len(valid) > 0:
            chunk_min, chunk_max = valid.min(), valid.max()
            min_date = chunk_min if min_date is None else min(min_date, chunk_min)
            max_date = chunk_max if max_date is None else max(max_date, chunk_max)

    return {
        'total_rows': total_rows,
        'bad_dates':  int(bad_dates),
        'min_date':   min_date,
        'max_date':   max_date,
    }


print('Scanning main file (this will take a few minutes)...')
summary_main = summarise_csv_dates(f'{TWITTER_RAW}/Bitcoin_tweets.csv')
print('\n--- Bitcoin_tweets.csv ---')
for k, v in summary_main.items():
    print(f'  {k:12s}: {v}')

print('\nScanning supplementary file...')
summary_2 = summarise_csv_dates(f'{TWITTER_RAW}/Bitcoin_tweets_dataset_2.csv')
print('\n--- Bitcoin_tweets_dataset_2.csv ---')
for k, v in summary_2.items():
    print(f'  {k:12s}: {v}')

In [ ]:
import pandas as pd

path = f'{TWITTER_RAW}/Bitcoin_tweets.csv'

# Stream the date column only, skipping malformed rows
total_rows = 0
bad_rows_skipped_proxy = 0  # rows the parser couldn't even read
min_date = None
max_date = None

for chunk in pd.read_csv(path,
                         usecols=['date'],
                         chunksize=100_000,
                         engine='python',
                         on_bad_lines='skip'):
    parsed = pd.to_datetime(chunk['date'], errors='coerce', utc=True)
    valid  = parsed.dropna()
    total_rows += len(chunk)
    if len(valid) > 0:
        chunk_min, chunk_max = valid.min(), valid.max()
        min_date = chunk_min if min_date is None else min(min_date, chunk_min)
        max_date = chunk_max if max_date is None else max(max_date, chunk_max)

print('--- Bitcoin_tweets.csv ---')
print(f'  Rows successfully read : {total_rows:,}')
print(f'  Earliest tweet date    : {min_date}')
print(f'  Latest tweet date      : {max_date}')

In [ ]:
import pandas as pd

path = f'{TWITTER_RAW}/Bitcoin_tweets_dataset_2.csv'

total_rows = 0
min_date = None
max_date = None

for chunk in pd.read_csv(path,
                         usecols=['date'],
                         chunksize=100_000,
                         engine='python',
                         on_bad_lines='skip'):
    parsed = pd.to_datetime(chunk['date'], errors='coerce', utc=True)
    valid  = parsed.dropna()
    total_rows += len(chunk)
    if len(valid) > 0:
        chunk_min, chunk_max = valid.min(), valid.max()
        min_date = chunk_min if min_date is None else min(min_date, chunk_min)
        max_date = chunk_max if max_date is None else max(max_date, chunk_max)

print('--- Bitcoin_tweets_dataset_2.csv ---')
print(f'  Rows successfully read : {total_rows:,}')
print(f'  Earliest tweet date    : {min_date}')
print(f'  Latest tweet date      : {max_date}')

In [ ]:
import pandas as pd

# Load the BTC prices we saved earlier
btc = pd.read_parquet(f'{PROJECT_ROOT}/data/raw/prices/btc_daily.parquet')
btc['date'] = pd.to_datetime(btc['date'])

# Trim to the Twitter window
start = pd.Timestamp('2021-02-05')
end   = pd.Timestamp('2023-01-09')
btc_trimmed = btc[(btc['date'] >= start) & (btc['date'] <= end)].copy()

print(f'Original BTC rows : {len(btc):,}')
print(f'Trimmed BTC rows  : {len(btc_trimmed):,}')
print(f'Date range        : {btc_trimmed.date.min().date()} → {btc_trimmed.date.max().date()}')

# Save the trimmed version
btc_trimmed.to_parquet(f'{PROJECT_ROOT}/data/processed/btc_daily_trimmed.parquet', index=False)
print('Saved trimmed BTC prices.')

In [ ]:
import numpy as np

btc = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/btc_daily_trimmed.parquet')
btc = btc.sort_values('date').reset_index(drop=True)

# Daily log returns
btc['log_return'] = np.log(btc['close'] / btc['close'].shift(1))

# Realised volatility: rolling 30-day std of log returns, annualised
WINDOW = 30
btc['realised_vol'] = btc['log_return'].rolling(window=WINDOW).std() * np.sqrt(365)

# Drop the warm-up period (first 30 rows where rolling window isn't full)
btc_vol = btc.dropna(subset=['realised_vol']).reset_index(drop=True)

print(f'Rows with valid volatility : {len(btc_vol):,}')
print(f'Date range                 : {btc_vol.date.min().date()} → {btc_vol.date.max().date()}')
print(f'Mean realised vol          : {btc_vol.realised_vol.mean():.3f}')
print(f'Min  / Max realised vol    : {btc_vol.realised_vol.min():.3f} / {btc_vol.realised_vol.max():.3f}')

btc_vol.to_parquet(f'{PROJECT_ROOT}/data/processed/btc_with_volatility.parquet', index=False)
print('Saved.')

In [ ]:
import matplotlib.pyplot as plt

btc_vol = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/btc_with_volatility.parquet')

fig, ax = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

ax[0].plot(btc_vol.date, btc_vol.close, color='#F7931A')
ax[0].set_title('BTC Close Price')
ax[0].set_ylabel('USDT')
ax[0].grid(True, alpha=0.3)

ax[1].plot(btc_vol.date, btc_vol.realised_vol, color='#cc3333')
ax[1].set_title('Realised Volatility (30-day rolling, annualised)')
ax[1].set_ylabel('Volatility')
ax[1].set_xlabel('Date')
ax[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
KEEP_COLS = ['date', 'text', 'user_followers', 'user_created',
             'user_friends', 'is_retweet']

path_in  = f'{TWITTER_RAW}/Bitcoin_tweets.csv'
path_out = f'{PROJECT_ROOT}/data/processed/tweets_clean.parquet'

chunks_out = []
rows_seen = 0
rows_kept = 0

for chunk in pd.read_csv(path_in,
                         usecols=KEEP_COLS,
                         chunksize=200_000,
                         engine='python',
                         on_bad_lines='skip'):
    rows_seen += len(chunk)

    # Parse date; drop rows where date is unparseable
    chunk['date'] = pd.to_datetime(chunk['date'], errors='coerce', utc=True)
    chunk = chunk.dropna(subset=['date', 'text'])

    # Drop retweets
    chunk = chunk[chunk['is_retweet'] == False].copy()

    # Restrict to study window
    chunk = chunk[(chunk['date'] >= '2021-02-05') &
                  (chunk['date'] <= '2023-01-09')]

    if len(chunk) > 0:
        chunks_out.append(chunk)
        rows_kept += len(chunk)

    # Progress indicator
    print(f'  Processed {rows_seen:,} rows, kept {rows_kept:,} so far', end='\r')

print()
print('Concatenating chunks...')
tweets = pd.concat(chunks_out, ignore_index=True)

print(f'\nFinal row count : {len(tweets):,}')
print(f'Date range      : {tweets.date.min().date()} → {tweets.date.max().date()}')

tweets.to_parquet(path_out, index=False)
print(f'Saved to {path_out}')

In [ ]:
# Force numeric columns to numeric; non-numeric values become NaN
tweets['user_followers'] = pd.to_numeric(tweets['user_followers'], errors='coerce')
tweets['user_friends']   = pd.to_numeric(tweets['user_friends'],   errors='coerce')

# Quick sanity check
print(tweets.dtypes)
print(f'\nRows: {len(tweets):,}')
print(f'user_followers NaN count: {tweets.user_followers.isna().sum():,}')
print(f'user_friends   NaN count: {tweets.user_friends.isna().sum():,}')

# Now save
tweets.to_parquet(path_out, index=False)
print(f'\nSaved to {path_out}')

In [ ]:
tweets = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/tweets_clean.parquet')

# Tweets per calendar day
tweets['day'] = tweets['date'].dt.date
daily_counts = tweets.groupby('day').size()

print(f'Days in study window     : {len(daily_counts):,}')
print(f'Mean tweets / day        : {daily_counts.mean():,.0f}')
print(f'Median tweets / day      : {daily_counts.median():,.0f}')
print(f'Min  tweets / day        : {daily_counts.min():,}')
print(f'Max  tweets / day        : {daily_counts.max():,}')
print(f'Days with <100 tweets    : {(daily_counts < 100).sum()}')
print(f'Days with <500 tweets    : {(daily_counts < 500).sum()}')

In [ ]:
import matplotlib.pyplot as plt

# Build a complete daily index for the full window
full_range = pd.date_range('2021-02-05', '2023-01-09', freq='D').date

# Build a series: count per day, with 0 for missing days
counts_full = pd.Series(0, index=full_range)
counts_full.update(daily_counts)

# Visualise
plt.figure(figsize=(14, 4))
plt.plot(counts_full.index, counts_full.values, linewidth=0.7)
plt.title('Tweets per day across study window (gaps = days with no coverage)')
plt.ylabel('Tweets')
plt.xlabel('Date')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Coverage by month
counts_full.index = pd.to_datetime(counts_full.index)
monthly_coverage = (counts_full > 0).resample('M').sum()
total_days_per_month = counts_full.resample('M').size()
coverage_pct = (monthly_coverage / total_days_per_month * 100).round(1)

print('\nMonthly coverage (% of days with tweets):')
for date, pct in coverage_pct.items():
    print(f'  {date.strftime("%Y-%m")} : {pct:5.1f}%')

In [ ]:
btc_vol = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/btc_with_volatility.parquet')
btc_vol['date'] = pd.to_datetime(btc_vol['date'])
btc_vol = btc_vol.set_index('date').sort_index()

# Weekly aggregation: week ending Sunday ('W-SUN')
# - close: last value of the week (Friday/Sunday close)
# - log_return: sum across the week (gives weekly log return)
# - realised_vol: mean over the week (average prevailing 30-day volatility)
weekly = btc_vol.resample('W-SUN').agg({
    'close': 'last',
    'log_return': 'sum',
    'realised_vol': 'mean',
})

# Drop any weeks that came out empty (shouldn't be any, but be safe)
weekly = weekly.dropna(subset=['realised_vol']).reset_index()

print(f'Weekly observations : {len(weekly)}')
print(f'Date range          : {weekly.date.min().date()} → {weekly.date.max().date()}')
print(f'Mean weekly vol     : {weekly.realised_vol.mean():.3f}')
print(f'Min  / Max          : {weekly.realised_vol.min():.3f} / {weekly.realised_vol.max():.3f}')

weekly.to_parquet(f'{PROJECT_ROOT}/data/processed/btc_weekly.parquet', index=False)
print('Saved.')

In [ ]:
!pip install -q emoji

In [ ]:
import re
import emoji

tweets = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/tweets_clean.parquet')
print(f'Loaded {len(tweets):,} tweets')

# Compiled regexes (compile once, reuse)
URL_RE     = re.compile(r'http\S+|www\.\S+')
MENTION_RE = re.compile(r'@\w+')
REPEAT_RE  = re.compile(r'(.)\1{2,}')   # 3+ repeated chars → 2 (mooooon → moon, but cool → cool)
WS_RE      = re.compile(r'\s+')

def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ''
    # 1. Convert emojis to :textual_descriptions:
    text = emoji.demojize(text, delimiters=(' :', ': '))
    # 2. Remove URLs
    text = URL_RE.sub(' ', text)
    # 3. Anonymise @mentions
    text = MENTION_RE.sub('@user', text)
    # 4. Collapse 3+ repeated characters down to 2
    text = REPEAT_RE.sub(r'\1\1', text)
    # 5. Normalise whitespace
    text = WS_RE.sub(' ', text).strip()
    return text

# Apply (this is the slow step — ~5–10 minutes on 4.66M rows)
print('Cleaning text (this will take a few minutes)...')
tweets['text_clean'] = tweets['text'].apply(clean_text)

# Drop tweets where cleaning left nothing meaningful (< 3 chars)
before = len(tweets)
tweets = tweets[tweets['text_clean'].str.len() >= 3].copy()
print(f'Dropped {before - len(tweets):,} tweets that cleaned to <3 chars')
print(f'Remaining: {len(tweets):,}')

# Sample to eyeball the cleaning
print('\n--- Sample cleaned tweets ---')
for i, row in tweets.sample(5, random_state=42).iterrows():
    print(f'\nORIGINAL: {row.text[:150]}')
    print(f'CLEANED : {row.text_clean[:150]}')

tweets.to_parquet(f'{PROJECT_ROOT}/data/processed/tweets_preprocessed.parquet', index=False)
print('\nSaved.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = '/content/drive/MyDrive/dissertation_cryptosent'
TWITTER_RAW  = f'{PROJECT_ROOT}/data/raw/twitter'
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
!pip install -q emoji

import pandas as pd
import re
import emoji
import os

In [ ]:
SRC  = f'{PROJECT_ROOT}/data/processed/tweets_clean.parquet'
DST  = f'{PROJECT_ROOT}/data/processed/tweets_preprocessed.parquet'
TMP_DIR = f'{PROJECT_ROOT}/data/processed/_tmp_chunks'
os.makedirs(TMP_DIR, exist_ok=True)

# Compiled regexes
URL_RE     = re.compile(r'http\S+|www\.\S+')
MENTION_RE = re.compile(r'@\w+')
REPEAT_RE  = re.compile(r'(.)\1{2,}')
WS_RE      = re.compile(r'\s+')

def clean_text(text):
    if not isinstance(text, str):
        return ''
    text = emoji.demojize(text, delimiters=(' :', ': '))
    text = URL_RE.sub(' ', text)
    text = MENTION_RE.sub('@user', text)
    text = REPEAT_RE.sub(r'\1\1', text)
    text = WS_RE.sub(' ', text).strip()
    return text

# Use pyarrow to read the parquet in batches without loading the whole thing
import pyarrow.parquet as pq

CHUNK_SIZE = 200_000
parquet_file = pq.ParquetFile(SRC)
total_rows = parquet_file.metadata.num_rows
print(f'Total rows to process: {total_rows:,}')

chunk_idx = 0
rows_processed = 0
rows_kept = 0

for batch in parquet_file.iter_batches(batch_size=CHUNK_SIZE):
    chunk = batch.to_pandas()
    chunk['text_clean'] = chunk['text'].apply(clean_text)
    chunk = chunk[chunk['text_clean'].str.len() >= 3]

    # Drop the original text column to save RAM/disk (we have the cleaned version)
    chunk = chunk.drop(columns=['text'])

    chunk.to_parquet(f'{TMP_DIR}/chunk_{chunk_idx:04d}.parquet', index=False)

    rows_processed += len(batch)
    rows_kept      += len(chunk)
    chunk_idx      += 1
    print(f'  Chunk {chunk_idx}: processed {rows_processed:,} / {total_rows:,}, kept {rows_kept:,}', end='\r')

    # Free memory explicitly
    del chunk, batch

print()
print(f'\nAll chunks written. Kept {rows_kept:,} tweets across {chunk_idx} files.')

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq
import glob

chunk_files = sorted(glob.glob(f'{TMP_DIR}/chunk_*.parquet'))
print(f'Found {len(chunk_files)} chunks to assemble')

# Stream-write the combined parquet using pyarrow's writer
schema = pq.read_schema(chunk_files[0])
with pq.ParquetWriter(DST, schema) as writer:
    for f in chunk_files:
        table = pq.read_table(f)
        writer.write_table(table)
        del table

print(f'Assembled file: {DST}')
print(f'Size on disk : {os.path.getsize(DST) / (1024**2):.1f} MB')

# Quick sanity check
sample = pd.read_parquet(DST, columns=['text_clean']).sample(5, random_state=42)
print('\n--- Sample cleaned tweets ---')
for t in sample.text_clean:
    print(f'  {t[:150]}')

# Clean up temp files (optional but tidy)
for f in chunk_files:
    os.remove(f)
os.rmdir(TMP_DIR)
print('\nTemp chunks cleaned up.')

In [ ]:
tweets = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/tweets_preprocessed.parquet')
print(f'Starting tweets: {len(tweets):,}')

# Parse user_created to datetime (currently a string)
tweets['user_created'] = pd.to_datetime(tweets['user_created'],
                                        errors='coerce', utc=True)

# ---------- Heuristic 1: Account age at time of posting < 30 days ----------
tweets['account_age_days'] = (tweets['date'] - tweets['user_created']).dt.days
flag_new_account = tweets['account_age_days'] < 30

# ---------- Heuristic 2: User posts > 100 tweets/day on average ----------
# Count tweets per user per day, then mark users who ever exceeded the threshold
user_day_counts = tweets.groupby([tweets['date'].dt.date, 'user_followers']).size()
# Actually, group by user identity (we don't have user_id, so use user_name as proxy)
# Note: this is approximate — user_name can change. user_id would be better but we
# don't have it. Document this in §4.3 as a limitation of the heuristic.
# Re-load with user_name for grouping:
tweets_meta = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/tweets_clean.parquet',
                              columns=['date', 'user_name'])
tweets_meta = tweets_meta.iloc[tweets.index] if len(tweets_meta) == len(tweets) else None

# Simpler & safer: re-pull user_name from the clean parquet aligned by row
# (the preprocessing step kept rows in order, so we can match)
tweets_with_user = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/tweets_clean.parquet',
                                   columns=['date', 'text', 'user_name', 'is_retweet'])
# Filter same way preprocessing did
tweets_with_user['date'] = pd.to_datetime(tweets_with_user['date'], errors='coerce', utc=True)
tweets_with_user = tweets_with_user.dropna(subset=['date', 'text'])
tweets_with_user = tweets_with_user[tweets_with_user['is_retweet'] == False]
tweets_with_user = tweets_with_user[(tweets_with_user['date'] >= '2021-02-05') &
                                    (tweets_with_user['date'] <= '2023-01-09')]
# Note: minor row-count drift from the <3-char drop is acceptable here for heuristic purposes
print(f'User-aligned frame: {len(tweets_with_user):,} rows')

# Posts per user per day
posts_per_user_per_day = tweets_with_user.groupby(
    [tweets_with_user['date'].dt.date, 'user_name']
).size().reset_index(name='count')

# Flag users who ever exceeded 100 posts/day
heavy_posters = set(
    posts_per_user_per_day[posts_per_user_per_day['count'] > 100]['user_name'].unique()
)
print(f'Heavy posters (>100/day on any day): {len(heavy_posters):,} unique users')

# Apply this flag back to the main tweets frame
tweets_with_user['flag_heavy_poster'] = tweets_with_user['user_name'].isin(heavy_posters)
# Reset index alignment with tweets dataframe
tweets = tweets.reset_index(drop=True)
tweets_with_user = tweets_with_user.reset_index(drop=True)
# Align by min length (defensive)
n = min(len(tweets), len(tweets_with_user))
tweets = tweets.iloc[:n].copy()
tweets['user_name'] = tweets_with_user['user_name'].iloc[:n].values
tweets['flag_heavy_poster'] = tweets_with_user['flag_heavy_poster'].iloc[:n].values

# ---------- Heuristic 3: Suspicious follower/friends ratios ----------
# Avoid division by zero
ratio = tweets['user_followers'] / tweets['user_friends'].replace(0, 1)
flag_auto_follower = (ratio < 0.1) & (tweets['user_friends'] > 100)
flag_no_audience   = tweets['user_followers'] < 5
flag_ratio = flag_auto_follower | flag_no_audience

# Re-evaluate Heuristic 1 since we re-indexed
flag_new_account = tweets['account_age_days'] < 30

# ---------- Combine all flags ----------
tweets['is_bot_suspected'] = (
    flag_new_account |
    tweets['flag_heavy_poster'] |
    flag_ratio
)

# Report
total = len(tweets)
print(f'\n--- Bot filter results ---')
print(f'  Tweets total                     : {total:,}')
print(f'  Flagged: new account (<30 days)  : {flag_new_account.sum():,}  ({flag_new_account.sum()/total*100:.1f}%)')
print(f'  Flagged: heavy poster (>100/day) : {tweets["flag_heavy_poster"].sum():,}  ({tweets["flag_heavy_poster"].sum()/total*100:.1f}%)')
print(f'  Flagged: suspicious ratio        : {flag_ratio.sum():,}  ({flag_ratio.sum()/total*100:.1f}%)')
print(f'  Flagged: ANY of the above        : {tweets["is_bot_suspected"].sum():,}  ({tweets["is_bot_suspected"].sum()/total*100:.1f}%)')

# Keep only non-bot tweets
tweets_filtered = tweets[~tweets['is_bot_suspected']].copy()
print(f'\n  Tweets after filtering           : {len(tweets_filtered):,}')
print(f'  Retention rate                   : {len(tweets_filtered)/total*100:.1f}%')

# Quick check: tweets per week after filtering
tweets_filtered['week'] = pd.to_datetime(tweets_filtered['date']).dt.to_period('W-SUN')
weekly_volume = tweets_filtered.groupby('week').size()
print(f'\n  Weeks with tweets                : {len(weekly_volume):,}')
print(f'  Median tweets per week           : {weekly_volume.median():,.0f}')
print(f'  Min    tweets per week           : {weekly_volume.min():,}')

# Save the filtered, preprocessed tweets
tweets_filtered.to_parquet(
    f'{PROJECT_ROOT}/data/processed/tweets_filtered.parquet',
    index=False
)
print(f'\nSaved filtered dataset.')

In [ ]:
tweets = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/tweets_preprocessed.parquet')
print(f'Starting tweets: {len(tweets):,}')

# Parse user_created to datetime (currently stored as string)
tweets['user_created'] = pd.to_datetime(tweets['user_created'],
                                        errors='coerce', utc=True)

# Compute account age at time of posting (in days)
tweets['account_age_days'] = (tweets['date'] - tweets['user_created']).dt.days

# Flag accounts younger than 30 days at the time of posting
tweets['flag_new_account'] = tweets['account_age_days'] < 30

# Report
n_flagged = tweets['flag_new_account'].sum()
print(f'\nHeuristic 1 — Account age < 30 days')
print(f'  Tweets flagged   : {n_flagged:,} ({n_flagged/len(tweets)*100:.2f}%)')
print(f'  Tweets unflagged : {(~tweets["flag_new_account"]).sum():,}')

# Also report a few diagnostics so we understand the distribution
print(f'\n  Account age distribution (days):')
print(f'    Min     : {tweets.account_age_days.min()}')
print(f'    Median  : {tweets.account_age_days.median():.0f}')
print(f'    Max     : {tweets.account_age_days.max()}')
print(f'    Negative: {(tweets.account_age_days < 0).sum():,}  (data errors)')

# Save with the flag added — don't drop anything yet
tweets.to_parquet(f'{PROJECT_ROOT}/data/processed/tweets_with_flags.parquet',
                  index=False)
print('\nSaved with flag_new_account column.')

In [ ]:
tweets = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/tweets_with_flags.parquet')
print(f'Loaded {len(tweets):,} tweets')

# Pattern 1: Auto-follower bot — follows lots, followed by few
# Guard against division by zero with .replace(0, 1)
ratio = tweets['user_followers'] / tweets['user_friends'].replace(0, 1)
flag_auto_follower = (ratio < 0.1) & (tweets['user_friends'] > 100)

# Pattern 2: No-audience account — <5 followers
flag_no_audience = tweets['user_followers'] < 5

# Combine: flag if EITHER pattern matches
tweets['flag_ratio'] = flag_auto_follower | flag_no_audience

# Report
n_auto    = flag_auto_follower.sum()
n_no_aud  = flag_no_audience.sum()
n_either  = tweets['flag_ratio'].sum()
n_both    = (flag_auto_follower & flag_no_audience).sum()

print(f'\nHeuristic 2 — Suspicious follower/friends ratio')
print(f'  Auto-follower pattern (ratio<0.1 & friends>100): {n_auto:,} ({n_auto/len(tweets)*100:.2f}%)')
print(f'  No-audience pattern   (followers<5)            : {n_no_aud:,} ({n_no_aud/len(tweets)*100:.2f}%)')
print(f'  Overlap (both patterns)                        : {n_both:,}')
print(f'  Flagged by EITHER pattern                      : {n_either:,} ({n_either/len(tweets)*100:.2f}%)')

# Distribution sanity check
print(f'\n  Follower distribution:')
print(f'    Min    : {tweets.user_followers.min():,.0f}')
print(f'    Median : {tweets.user_followers.median():,.0f}')
print(f'    Max    : {tweets.user_followers.max():,.0f}')

# Save updated flags
tweets.to_parquet(f'{PROJECT_ROOT}/data/processed/tweets_with_flags.parquet',
                  index=False)
print('\nSaved with flag_ratio column added.')

In [ ]:
print('Re-pulling user_name from source CSV...')

user_name_chunks = []
for chunk in pd.read_csv(f'{TWITTER_RAW}/Bitcoin_tweets.csv',
                         usecols=['date', 'text', 'user_name', 'is_retweet'],
                         chunksize=200_000,
                         engine='python',
                         on_bad_lines='skip'):
    chunk['date'] = pd.to_datetime(chunk['date'], errors='coerce', utc=True)
    chunk = chunk.dropna(subset=['date', 'text'])
    chunk = chunk[chunk['is_retweet'] == False]
    chunk = chunk[(chunk['date'] >= '2021-02-05') &
                  (chunk['date'] <= '2023-01-09')]
    user_name_chunks.append(chunk[['user_name']].reset_index(drop=True))

user_names_df = pd.concat(user_name_chunks, ignore_index=True)
print(f'Pulled {len(user_names_df):,} user_name rows')

# Now load the flagged tweets and check alignment
tweets = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/tweets_with_flags.parquet')
print(f'Tweets in flags file : {len(tweets):,}')
print(f'User_names pulled    : {len(user_names_df):,}')
print(f'Difference           : {len(user_names_df) - len(tweets):,}')

In [ ]:
# Align user_names to tweets (truncate the 2-row excess)
user_names_aligned = user_names_df.iloc[:len(tweets)].reset_index(drop=True)
tweets = tweets.reset_index(drop=True)
tweets['user_name'] = user_names_aligned['user_name'].values

print(f'Attached user_name. NaN user_names: {tweets.user_name.isna().sum():,}')

# Posts per user per day
posts_per_user_per_day = tweets.groupby(
    [tweets['date'].dt.date, 'user_name']
).size().reset_index(name='count')

# Find users who ever exceeded 100 posts in a single day
heavy_posters = set(
    posts_per_user_per_day[posts_per_user_per_day['count'] > 100]['user_name'].unique()
)
print(f'\nHeuristic 3 — Heavy posters (>100 tweets on any single day)')
print(f'  Unique users flagged as heavy posters: {len(heavy_posters):,}')

# Flag all tweets from those users
tweets['flag_heavy_poster'] = tweets['user_name'].isin(heavy_posters)
n_heavy = tweets['flag_heavy_poster'].sum()
print(f'  Total tweets flagged                 : {n_heavy:,} ({n_heavy/len(tweets)*100:.2f}%)')

# Save updated flags
tweets.to_parquet(f'{PROJECT_ROOT}/data/processed/tweets_with_flags.parquet',
                  index=False)
print('\nSaved with flag_heavy_poster column added.')

In [ ]:
tweets = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/tweets_with_flags.parquet')

# Combine all three flags — OR logic (flagged if ANY heuristic catches it)
tweets['is_bot_suspected'] = (
    tweets['flag_new_account'] |
    tweets['flag_ratio'] |
    tweets['flag_heavy_poster']
)

total = len(tweets)
print(f'Combined bot filter results')
print(f'  Total tweets                     : {total:,}')
print(f'  Flagged: new account             : {tweets["flag_new_account"].sum():,}  ({tweets["flag_new_account"].sum()/total*100:5.2f}%)')
print(f'  Flagged: suspicious ratio        : {tweets["flag_ratio"].sum():,}  ({tweets["flag_ratio"].sum()/total*100:5.2f}%)')
print(f'  Flagged: heavy poster            : {tweets["flag_heavy_poster"].sum():,}  ({tweets["flag_heavy_poster"].sum()/total*100:5.2f}%)')
print(f'  Flagged: ANY heuristic           : {tweets["is_bot_suspected"].sum():,}  ({tweets["is_bot_suspected"].sum()/total*100:5.2f}%)')

# Overlap matrix — how much do the heuristics agree?
print(f'\n  Pairwise overlap:')
print(f'    New account ∩ Ratio          : {(tweets.flag_new_account & tweets.flag_ratio).sum():,}')
print(f'    New account ∩ Heavy poster   : {(tweets.flag_new_account & tweets.flag_heavy_poster).sum():,}')
print(f'    Ratio       ∩ Heavy poster   : {(tweets.flag_ratio & tweets.flag_heavy_poster).sum():,}')
print(f'    All three                    : {(tweets.flag_new_account & tweets.flag_ratio & tweets.flag_heavy_poster).sum():,}')

# Filter to non-bot tweets
tweets_filtered = tweets[~tweets['is_bot_suspected']].copy()
print(f'\n  Tweets after bot filtering       : {len(tweets_filtered):,}')
print(f'  Retention rate                   : {len(tweets_filtered)/total*100:.2f}%')

# Sanity check: weekly volume after filtering
tweets_filtered['week'] = pd.to_datetime(tweets_filtered['date']).dt.to_period('W-SUN')
weekly_volume = tweets_filtered.groupby('week').size()
print(f'\n  Weeks with tweets                : {len(weekly_volume):,}')
print(f'  Median tweets per week           : {weekly_volume.median():,.0f}')
print(f'  Min    tweets per week           : {weekly_volume.min():,}')
print(f'  Max    tweets per week           : {weekly_volume.max():,}')

# Save final filtered dataset
tweets_filtered.to_parquet(
    f'{PROJECT_ROOT}/data/processed/tweets_filtered.parquet',
    index=False
)
print(f'\nSaved final filtered dataset.')

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import copy

# Standard VADER (unmodified — the baseline)
vader_standard = SentimentIntensityAnalyzer()

# Crypto-augmented VADER — start from a deep copy, then extend the lexicon
vader_crypto = SentimentIntensityAnalyzer()

CRYPTO_LEXICON = {
    # Bullish vocabulary
    'hodl':         1.8,
    'hodling':      1.8,
    'hodler':       1.5,
    'moon':         2.5,
    'mooning':      2.5,
    'mooned':       2.0,
    'bullish':      2.0,
    'bull':         1.5,
    'bullrun':      2.2,
    'pump':         1.5,
    'pumping':      1.5,
    'pumped':       1.2,
    'lambo':        1.8,
    'wagmi':        2.0,
    'diamondhands': 1.8,   # collapsed from "diamond hands"
    'wenmoon':      1.5,
    'gigabullish':  2.8,

    # Bearish vocabulary
    'bearish':      -2.0,
    'bear':         -1.2,
    'dump':         -1.5,
    'dumping':      -1.5,
    'dumped':       -1.2,
    'rekt':         -2.8,
    'fud':          -2.0,
    'fomo':         -1.0,
    'shill':        -1.5,
    'shilling':     -1.5,
    'shitcoin':     -2.5,
    'scam':         -3.0,
    'scammer':      -3.0,
    'rugpull':      -3.2,
    'rugged':       -3.0,
    'paperhands':   -1.5,   # collapsed from "paper hands"
    'bagholder':    -1.8,
    'bagholding':   -1.8,
    'ngmi':         -2.0,
    'cope':         -1.0,
    'capitulate':   -2.0,
    'capitulation': -2.0,

    # Mildly bullish/bearish or ambiguous
    'whale':        0.5,
    'whales':       0.5,
    'degen':        0.3,
    'normie':       -0.3,
}

# Update the augmented analyser's lexicon
vader_crypto.lexicon.update(CRYPTO_LEXICON)

print(f'Standard VADER lexicon size : {len(vader_standard.lexicon):,}')
print(f'Crypto VADER lexicon size   : {len(vader_crypto.lexicon):,}')
print(f'Added crypto terms          : {len(CRYPTO_LEXICON)}')

# Quick demo — same tweet, two analysers
demo_tweet = "BTC mooning hard! HODL through the FUD, diamondhands only. wagmi 🚀"
print(f'\n--- Demo on the same tweet ---')
print(f'Tweet: "{demo_tweet}"')
print(f'  Standard VADER : {vader_standard.polarity_scores(demo_tweet)["compound"]:+.4f}')
print(f'  Crypto  VADER  : {vader_crypto.polarity_scores(demo_tweet)["compound"]:+.4f}')

I think when i did skip the high-RAM earlier it restarted the session maybe this is why installing VADER was wiped.

In [ ]:
!pip install -q vaderSentiment

In [ ]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import copy

# Standard VADER (unmodified — the baseline)
vader_standard = SentimentIntensityAnalyzer()

# Crypto-augmented VADER — start from a deep copy, then extend the lexicon
vader_crypto = SentimentIntensityAnalyzer()

CRYPTO_LEXICON = {
    # Bullish vocabulary
    'hodl':         1.8,
    'hodling':      1.8,
    'hodler':       1.5,
    'moon':         2.5,
    'mooning':      2.5,
    'mooned':       2.0,
    'bullish':      2.0,
    'bull':         1.5,
    'bullrun':      2.2,
    'pump':         1.5,
    'pumping':      1.5,
    'pumped':       1.2,
    'lambo':        1.8,
    'wagmi':        2.0,
    'diamondhands': 1.8,   # collapsed from "diamond hands"
    'wenmoon':      1.5,
    'gigabullish':  2.8,

    # Bearish vocabulary
    'bearish':      -2.0,
    'bear':         -1.2,
    'dump':         -1.5,
    'dumping':      -1.5,
    'dumped':       -1.2,
    'rekt':         -2.8,
    'fud':          -2.0,
    'fomo':         -1.0,
    'shill':        -1.5,
    'shilling':     -1.5,
    'shitcoin':     -2.5,
    'scam':         -3.0,
    'scammer':      -3.0,
    'rugpull':      -3.2,
    'rugged':       -3.0,
    'paperhands':   -1.5,   # collapsed from "paper hands"
    'bagholder':    -1.8,
    'bagholding':   -1.8,
    'ngmi':         -2.0,
    'cope':         -1.0,
    'capitulate':   -2.0,
    'capitulation': -2.0,

    # Mildly bullish/bearish or ambiguous
    'whale':        0.5,
    'whales':       0.5,
    'degen':        0.3,
    'normie':       -0.3,
}

# Update the augmented analyser's lexicon
vader_crypto.lexicon.update(CRYPTO_LEXICON)

print(f'Standard VADER lexicon size : {len(vader_standard.lexicon):,}')
print(f'Crypto VADER lexicon size   : {len(vader_crypto.lexicon):,}')
print(f'Added crypto terms          : {len(CRYPTO_LEXICON)}')

# Quick demo — same tweet, two analysers
demo_tweet = "BTC mooning hard! HODL through the FUD, diamondhands only. wagmi 🚀"
print(f'\n--- Demo on the same tweet ---')
print(f'Tweet: "{demo_tweet}"')
print(f'  Standard VADER : {vader_standard.polarity_scores(demo_tweet)["compound"]:+.4f}')
print(f'  Crypto  VADER  : {vader_crypto.polarity_scores(demo_tweet)["compound"]:+.4f}')

In [ ]:
import time

tweets = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/tweets_filtered.parquet')
print(f'Scoring {len(tweets):,} tweets...')

start = time.time()

# Score with both analysers
texts = tweets['text_clean'].values  # numpy array is faster to iterate
vader_std_scores    = [vader_standard.polarity_scores(t)['compound'] for t in texts]
print(f'  Standard VADER done   ({time.time()-start:.1f}s)')

vader_crypto_scores = [vader_crypto.polarity_scores(t)['compound'] for t in texts]
print(f'  Crypto   VADER done   ({time.time()-start:.1f}s)')

tweets['vader_std']    = vader_std_scores
tweets['vader_crypto'] = vader_crypto_scores

# Quick distribution summary
print(f'\n--- Score distributions ---')
print(f'  Standard VADER:  mean={tweets.vader_std.mean():+.4f}   '
      f'median={tweets.vader_std.median():+.4f}   '
      f'std={tweets.vader_std.std():.4f}')
print(f'  Crypto   VADER:  mean={tweets.vader_crypto.mean():+.4f}   '
      f'median={tweets.vader_crypto.median():+.4f}   '
      f'std={tweets.vader_crypto.std():.4f}')

# Divergence between the two methods
tweets['vader_diff'] = tweets['vader_crypto'] - tweets['vader_std']
print(f'\n  Mean absolute difference (crypto − standard): {tweets.vader_diff.abs().mean():.4f}')
print(f'  Tweets where the two methods give opposite signs: '
      f'{((tweets.vader_std * tweets.vader_crypto) < 0).sum():,} '
      f'({((tweets.vader_std * tweets.vader_crypto) < 0).mean()*100:.2f}%)')

# Save
tweets.to_parquet(f'{PROJECT_ROOT}/data/processed/tweets_vader.parquet', index=False)
print(f'\nSaved to tweets_vader.parquet')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/dissertation_cryptosent'
TWITTER_RAW  = f'{PROJECT_ROOT}/data/raw/twitter'

import pandas as pd
import numpy as np
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import torch
print(f'PyTorch version : {torch.__version__}')
print(f'CUDA available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU device      : {torch.cuda.get_device_name(0)}')
    print(f'GPU memory      : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
else:
    print('⚠️  GPU NOT AVAILABLE — go to Runtime → Change runtime type → T4 GPU')

In [ ]:
!pip install -q transformers

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_NAME = 'ProsusAI/finbert'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Loading FinBERT tokeniser and model...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()  # inference mode — disables dropout, etc.

print(f'Loaded on device: {DEVICE}')
print(f'Model labels    : {model.config.id2label}')

In [ ]:
import torch
import torch.nn.functional as F
import time

# Load filtered tweets
tweets = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/tweets_vader.parquet')
print(f'Total tweets to score: {len(tweets):,}')

# TEST RUN: just the first 1,000 tweets
test_texts = tweets['text_clean'].head(1000).tolist()

BATCH_SIZE = 32
MAX_LENGTH = 256  # FinBERT supports up to 512, but most tweets are far shorter;
                  # 256 is 2x faster and covers >99% of tweets without truncation

def score_finbert_batch(texts: list, batch_size: int = BATCH_SIZE) -> np.ndarray:
    """
    Score a list of texts with FinBERT.
    Returns a numpy array of shape (len(texts), 3) with [P_pos, P_neg, P_neu].
    """
    all_probs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]

        # Tokenise the batch — pad shorter texts to the longest in the batch,
        # truncate longer ones to MAX_LENGTH
        inputs = tokenizer(batch, padding=True, truncation=True,
                          max_length=MAX_LENGTH, return_tensors='pt').to(DEVICE)

        # Inference without gradient tracking
        with torch.no_grad():
            outputs = model(**inputs)

        # Convert logits to probabilities via softmax
        probs = F.softmax(outputs.logits, dim=-1).cpu().numpy()
        all_probs.append(probs)

    return np.vstack(all_probs)

# Time the test run
print(f'\nTest run on {len(test_texts):,} tweets, batch size {BATCH_SIZE}...')
start = time.time()
probs = score_finbert_batch(test_texts)
elapsed = time.time() - start

# Extract individual probabilities (column order matches model.config.id2label)
# {0: 'positive', 1: 'negative', 2: 'neutral'}
p_pos = probs[:, 0]
p_neg = probs[:, 1]
p_neu = probs[:, 2]

# Continuous score
finbert_score = p_pos - p_neg

print(f'\n--- Test results ---')
print(f'  Tweets scored        : {len(test_texts):,}')
print(f'  Time elapsed         : {elapsed:.1f} seconds')
print(f'  Speed                : {len(test_texts)/elapsed:.0f} tweets/sec')
print(f'  Estimated time for {len(tweets):,} tweets : '
      f'{len(tweets)/(len(test_texts)/elapsed)/3600:.1f} hours')
print(f'\n  Score statistics:')
print(f'    Mean    : {finbert_score.mean():+.4f}')
print(f'    Median  : {np.median(finbert_score):+.4f}')
print(f'    Std     : {finbert_score.std():.4f}')
print(f'    Min/Max : {finbert_score.min():+.4f} / {finbert_score.max():+.4f}')

# Eyeball a few examples
print(f'\n--- Sample tweets and their FinBERT scores ---')
for idx in [0, 100, 500, 999]:
    text_preview = test_texts[idx][:120]
    print(f'\n  [{idx}] "{text_preview}..."')
    print(f'        pos={p_pos[idx]:.3f}  neg={p_neg[idx]:.3f}  neu={p_neu[idx]:.3f}  '
          f'→ score={finbert_score[idx]:+.4f}')

In [ ]:
import os
import time

# Where to save the chunked FinBERT results
FINBERT_CHUNKS_DIR = f'{PROJECT_ROOT}/data/processed/_finbert_chunks'
os.makedirs(FINBERT_CHUNKS_DIR, exist_ok=True)

CHUNK_SIZE = 50_000
total_tweets = len(tweets)
n_chunks = (total_tweets + CHUNK_SIZE - 1) // CHUNK_SIZE

print(f'Total tweets to score: {total_tweets:,}')
print(f'Chunk size           : {CHUNK_SIZE:,}')
print(f'Total chunks         : {n_chunks}')
print(f'Estimated total time : {total_tweets / 191 / 3600:.1f} hours\n')

# Check for completed chunks (resume support)
completed = set()
for f in os.listdir(FINBERT_CHUNKS_DIR):
    if f.startswith('chunk_') and f.endswith('.npy'):
        idx = int(f.replace('chunk_', '').replace('.npy', ''))
        completed.add(idx)

if completed:
    print(f'Resuming: found {len(completed)} completed chunks. Skipping those.\n')

# Process all chunks
all_texts = tweets['text_clean'].values
overall_start = time.time()

for chunk_idx in range(n_chunks):
    if chunk_idx in completed:
        continue

    chunk_start = chunk_idx * CHUNK_SIZE
    chunk_end   = min(chunk_start + CHUNK_SIZE, total_tweets)
    chunk_texts = all_texts[chunk_start:chunk_end].tolist()

    t0 = time.time()
    probs = score_finbert_batch(chunk_texts)
    elapsed = time.time() - t0

    # Save this chunk
    np.save(f'{FINBERT_CHUNKS_DIR}/chunk_{chunk_idx:04d}.npy', probs)

    # Progress report
    done_so_far = sum(1 for i in range(chunk_idx + 1) if i in completed or i == chunk_idx)
    total_elapsed = time.time() - overall_start
    remaining_chunks = n_chunks - chunk_idx - 1
    eta_seconds = (total_elapsed / (chunk_idx + 1 - len(completed))) * remaining_chunks if remaining_chunks > 0 else 0

    print(f'  Chunk {chunk_idx+1}/{n_chunks} done in {elapsed:.0f}s '
          f'({len(chunk_texts)/elapsed:.0f} tweets/sec). '
          f'ETA: {eta_seconds/3600:.1f}h')

print(f'\n✅ All chunks complete. Total time: {(time.time()-overall_start)/3600:.2f} hours')

In [ ]:
import glob

# Find all chunk files, sorted
chunk_files = sorted(glob.glob(f'{FINBERT_CHUNKS_DIR}/chunk_*.npy'))
print(f'Found {len(chunk_files)} chunk files')

# Load and stack
all_probs = np.vstack([np.load(f) for f in chunk_files])
print(f'Combined shape: {all_probs.shape}')

# Load the tweets dataframe that has VADER scores
tweets = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/tweets_vader.parquet')
print(f'Tweets dataframe rows: {len(tweets):,}')

# Sanity check alignment
assert len(all_probs) == len(tweets), \
    f'Mismatch: {len(all_probs)} probs vs {len(tweets)} tweets'

# Attach as columns
# Label order from earlier: {0: 'positive', 1: 'negative', 2: 'neutral'}
tweets['finbert_pos']   = all_probs[:, 0]
tweets['finbert_neg']   = all_probs[:, 1]
tweets['finbert_neu']   = all_probs[:, 2]
tweets['finbert_score'] = tweets['finbert_pos'] - tweets['finbert_neg']

# Distribution summary
print(f'\n--- FinBERT score distribution (full corpus) ---')
print(f'  Mean    : {tweets.finbert_score.mean():+.4f}')
print(f'  Median  : {tweets.finbert_score.median():+.4f}')
print(f'  Std     : {tweets.finbert_score.std():.4f}')
print(f'  Min/Max : {tweets.finbert_score.min():+.4f} / {tweets.finbert_score.max():+.4f}')

# Cross-method comparison: how often do FinBERT and VADER agree on sign?
both_pos = ((tweets.finbert_score > 0.05) & (tweets.vader_std > 0.05)).sum()
both_neg = ((tweets.finbert_score < -0.05) & (tweets.vader_std < -0.05)).sum()
both_neu = ((tweets.finbert_score.abs() <= 0.05) & (tweets.vader_std.abs() <= 0.05)).sum()
disagree = len(tweets) - both_pos - both_neg - both_neu

print(f'\n--- FinBERT vs Standard VADER agreement ---')
print(f'  Both positive  : {both_pos:,} ({both_pos/len(tweets)*100:.1f}%)')
print(f'  Both negative  : {both_neg:,} ({both_neg/len(tweets)*100:.1f}%)')
print(f'  Both neutral   : {both_neu:,} ({both_neu/len(tweets)*100:.1f}%)')
print(f'  Disagree       : {disagree:,} ({disagree/len(tweets)*100:.1f}%)')

# Save
tweets.to_parquet(f'{PROJECT_ROOT}/data/processed/tweets_scored.parquet', index=False)
print(f'\nSaved tweets_scored.parquet')

In [ ]:
tweets = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/tweets_scored.parquet')
print(f'Loaded {len(tweets):,} tweets')

# Compute log-followers weight column
tweets['weight'] = np.log(tweets['user_followers'].fillna(0) + 2)

# Assign each tweet to a week-ending-Sunday period
tweets['date'] = pd.to_datetime(tweets['date'])
tweets['week'] = tweets['date'].dt.to_period('W-SUN').dt.start_time + pd.Timedelta(days=6)
# week column now contains the Sunday end-date of each week

def weighted_mean(values, weights):
    """Weighted mean, safe against zero-weight weeks."""
    w_sum = weights.sum()
    if w_sum == 0:
        return values.mean()
    return (values * weights).sum() / w_sum

# Weekly aggregation
print('Aggregating to weekly...')
agg_records = []
for week, group in tweets.groupby('week'):
    agg_records.append({
        'week': week,
        'tweet_count':           len(group),
        # Simple means
        'vader_std_mean':        group['vader_std'].mean(),
        'vader_crypto_mean':     group['vader_crypto'].mean(),
        'finbert_mean':          group['finbert_score'].mean(),
        # Follower-weighted means
        'vader_std_wmean':       weighted_mean(group['vader_std'],    group['weight']),
        'vader_crypto_wmean':    weighted_mean(group['vader_crypto'], group['weight']),
        'finbert_wmean':         weighted_mean(group['finbert_score'], group['weight']),
        # Diagnostic — total follower weight (proxy for total reach)
        'total_log_followers':   group['weight'].sum(),
    })

weekly_sent = pd.DataFrame(agg_records).sort_values('week').reset_index(drop=True)

# Quick distribution check
print(f'\nWeekly observations : {len(weekly_sent)}')
print(f'Date range          : {weekly_sent.week.min().date()} → {weekly_sent.week.max().date()}')
print(f'\nTweets per week:')
print(f'  Min    : {weekly_sent.tweet_count.min():,}')
print(f'  Median : {weekly_sent.tweet_count.median():,.0f}')
print(f'  Max    : {weekly_sent.tweet_count.max():,}')

print(f'\nMean sentiment scores (across weeks):')
print(f'  VADER std        : {weekly_sent.vader_std_mean.mean():+.4f}')
print(f'  VADER crypto     : {weekly_sent.vader_crypto_mean.mean():+.4f}')
print(f'  FinBERT          : {weekly_sent.finbert_mean.mean():+.4f}')

print(f'\nMean follower-weighted sentiment (across weeks):')
print(f'  VADER std (w)    : {weekly_sent.vader_std_wmean.mean():+.4f}')
print(f'  VADER crypto (w) : {weekly_sent.vader_crypto_wmean.mean():+.4f}')
print(f'  FinBERT (w)      : {weekly_sent.finbert_wmean.mean():+.4f}')

# Save
weekly_sent.to_parquet(f'{PROJECT_ROOT}/data/processed/weekly_sentiment.parquet', index=False)
print(f'\nSaved weekly_sentiment.parquet')

In [ ]:
# Load both
btc_weekly  = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/btc_weekly.parquet')
sent_weekly = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/weekly_sentiment.parquet')

print(f'BTC weekly rows       : {len(btc_weekly)}')
print(f'Sentiment weekly rows : {len(sent_weekly)}')

# Normalise both date columns to the same type
btc_weekly['date']  = pd.to_datetime(btc_weekly['date']).dt.tz_localize(None).dt.normalize()
sent_weekly['week'] = pd.to_datetime(sent_weekly['week']).dt.tz_localize(None).dt.normalize()

# Inner join on date
merged = btc_weekly.merge(sent_weekly, left_on='date', right_on='week', how='inner')
merged = merged.drop(columns=['week'])

print(f'\nMerged rows           : {len(merged)}')
print(f'Date range            : {merged.date.min().date()} → {merged.date.max().date()}')
print(f'\nColumns: {list(merged.columns)}')

# Save the final aligned table
merged.to_parquet(f'{PROJECT_ROOT}/data/processed/weekly_features.parquet', index=False)
print(f'\nSaved weekly_features.parquet')

# Quick look at the data
print('\n--- First 3 rows ---')
print(merged.head(3).to_string())

In [ ]:
df = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/weekly_features.parquet')
df = df.sort_values('date').reset_index(drop=True)

# --- Price-derived features (baseline) ---
df['vol_lag1'] = df['realised_vol'].shift(1)
df['vol_lag2'] = df['realised_vol'].shift(2)
df['vol_lag4'] = df['realised_vol'].shift(4)

df['return_lag1']     = df['log_return'].shift(1)
df['abs_return_lag1'] = df['log_return'].abs().shift(1)

# Rolling stats (using only past data — so we shift the result)
df['return_std_4w']   = df['log_return'].rolling(4).std().shift(1)
df['return_mean_4w']  = df['log_return'].rolling(4).mean().shift(1)

# --- Sentiment-derived features (the experimental addition) ---
for col in ['vader_std_mean', 'vader_crypto_mean', 'finbert_mean',
            'vader_std_wmean', 'vader_crypto_wmean', 'finbert_wmean',
            'tweet_count']:
    df[f'{col}_lag1'] = df[col].shift(1)

# Drop rows with NaN from the lagging (the first 4 weeks)
df_features = df.dropna().reset_index(drop=True)

print(f'Original rows : {len(df)}')
print(f'After lagging : {len(df_features)} (dropped first 4 weeks for lag warmup)')
print(f'Date range    : {df_features.date.min().date()} → {df_features.date.max().date()}')
print(f'\nFeature columns ({len(df_features.columns)}):')
for c in df_features.columns:
    print(f'  {c}')

# Save
df_features.to_parquet(f'{PROJECT_ROOT}/data/processed/weekly_features_final.parquet',
                       index=False)
print(f'\nSaved weekly_features_final.parquet')

In [ ]:
!pip install -q prophet

In [ ]:
from prophet import Prophet
import warnings
warnings.filterwarnings('ignore')  # Prophet/cmdstanpy is chatty

df = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/weekly_features_final.parquet')
print(f'Loaded {len(df)} weekly rows')

# Prophet format: ds (date), y (target), + regressor columns
df_p = df.rename(columns={'date': 'ds', 'realised_vol': 'y'}).copy()
df_p['ds'] = pd.to_datetime(df_p['ds'])

# Price-only regressors (the baseline feature set)
PRICE_FEATURES = ['vol_lag1', 'vol_lag2', 'vol_lag4',
                  'return_lag1', 'abs_return_lag1',
                  'return_std_4w', 'return_mean_4w']

# Train/test split — use last 8 weeks for test
n_test = 8
train = df_p.iloc[:-n_test].copy()
test  = df_p.iloc[-n_test:].copy()
print(f'Train weeks: {len(train)}')
print(f'Test weeks : {len(test)}  ({test.ds.min().date()} → {test.ds.max().date()})')

# Build Prophet model with price regressors
m = Prophet(
    yearly_seasonality=False,
    weekly_seasonality=False,
    daily_seasonality=False,
    growth='linear',
    changepoint_prior_scale=0.05,  # default — controls how flexible trend is
)
for feat in PRICE_FEATURES:
    m.add_regressor(feat)

# Fit
print('\nFitting Prophet baseline...')
m.fit(train[['ds', 'y'] + PRICE_FEATURES])

# Forecast on test set
forecast = m.predict(test[['ds'] + PRICE_FEATURES])
test['y_pred'] = forecast['yhat'].values

# Evaluate
from sklearn.metrics import mean_squared_error, mean_absolute_error
rmse = mean_squared_error(test['y'], test['y_pred']) ** 0.5
mae  = mean_absolute_error(test['y'], test['y_pred'])

# Directional accuracy: did the model correctly predict whether vol went up or down?
test['y_change']      = test['y'].diff()
test['y_pred_change'] = test['y_pred'].diff()
direction_correct = (np.sign(test['y_change']) == np.sign(test['y_pred_change']))
da = direction_correct.iloc[1:].mean()  # skip first row (no prior to diff against)

print(f'\n--- Prophet Baseline (price-only) Test Results ---')
print(f'  RMSE                : {rmse:.4f}')
print(f'  MAE                 : {mae:.4f}')
print(f'  Directional accuracy: {da:.2%}')
print(f'\nTest predictions:')
for _, r in test.iterrows():
    print(f'  {r.ds.date()}  actual={r.y:.3f}  predicted={r.y_pred:.3f}  '
          f'error={r.y - r.y_pred:+.3f}')

# Save predictions for later comparison
test[['ds', 'y', 'y_pred']].to_parquet(
    f'{PROJECT_ROOT}/data/processed/preds_prophet_baseline.parquet', index=False
)
print('\nSaved baseline predictions.')

In [ ]:
SENT_VARIANTS = {
    'vader_std':       'vader_std_mean_lag1',
    'vader_std_w':     'vader_std_wmean_lag1',
    'vader_crypto':    'vader_crypto_mean_lag1',
    'vader_crypto_w':  'vader_crypto_wmean_lag1',
    'finbert':         'finbert_mean_lag1',
    'finbert_w':       'finbert_wmean_lag1',
}

results = []
all_predictions = {}

for variant_name, sent_feat in SENT_VARIANTS.items():
    feature_set = PRICE_FEATURES + [sent_feat, 'tweet_count_lag1']

    m = Prophet(
        yearly_seasonality=False,
        weekly_seasonality=False,
        daily_seasonality=False,
        growth='linear',
        changepoint_prior_scale=0.05,
    )
    for feat in feature_set:
        m.add_regressor(feat)

    m.fit(train[['ds', 'y'] + feature_set])
    forecast = m.predict(test[['ds'] + feature_set])

    pred = forecast['yhat'].values
    rmse = mean_squared_error(test['y'], pred) ** 0.5
    mae  = mean_absolute_error(test['y'], pred)

    # Directional accuracy
    pred_change = np.diff(pred)
    actual_change = test['y'].diff().iloc[1:].values
    da = (np.sign(pred_change) == np.sign(actual_change)).mean()

    results.append({
        'variant': variant_name,
        'rmse': rmse,
        'mae':  mae,
        'da':   da,
    })
    all_predictions[variant_name] = pred

# Print comparison table
print(f'\n{"Variant":<20s}  {"RMSE":>7s}  {"MAE":>7s}  {"DA":>7s}  {"Δ vs base":>10s}')
print('-' * 60)
baseline_rmse = 0.1826  # from previous cell
print(f'{"baseline (price)":<20s}  {baseline_rmse:>7.4f}  {0.1419:>7.4f}  {0.7143:>7.2%}  {"":>10s}')
for r in results:
    delta = (r['rmse'] - baseline_rmse) / baseline_rmse * 100
    sign  = '↓' if delta < 0 else '↑'
    print(f'{r["variant"]:<20s}  {r["rmse"]:>7.4f}  {r["mae"]:>7.4f}  {r["da"]:>7.2%}  {sign}{abs(delta):>6.2f}%')

# Save all predictions for later DM testing
pred_df = pd.DataFrame({'ds': test['ds'].values, 'y_true': test['y'].values})
pred_df['baseline'] = test['y_pred'].values
for variant, pred in all_predictions.items():
    pred_df[variant] = pred
pred_df.to_parquet(f'{PROJECT_ROOT}/data/processed/preds_prophet_all.parquet', index=False)
print('\nSaved all predictions.')

In [ ]:
from prophet import Prophet
import warnings
warnings.filterwarnings('ignore')
import logging
logging.getLogger('prophet').setLevel(logging.WARNING)
logging.getLogger('cmdstanpy').setLevel(logging.WARNING)

df = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/weekly_features_final.parquet')
df_p = df.rename(columns={'date': 'ds', 'realised_vol': 'y'}).copy()
df_p['ds'] = pd.to_datetime(df_p['ds'])
print(f'Total weeks available: {len(df_p)}')

# Walk-forward configuration
MIN_TRAIN_SIZE = 25   # need enough weeks for Prophet to learn
N_FOLDS = len(df_p) - MIN_TRAIN_SIZE
print(f'Walk-forward folds: {N_FOLDS} (from week {MIN_TRAIN_SIZE} to week {len(df_p)-1})')

# Variants to evaluate
PRICE_FEATURES = ['vol_lag1', 'vol_lag2', 'vol_lag4',
                  'return_lag1', 'abs_return_lag1',
                  'return_std_4w', 'return_mean_4w']

VARIANTS = {
    'baseline':        PRICE_FEATURES,
    'vader_std':       PRICE_FEATURES + ['vader_std_mean_lag1',    'tweet_count_lag1'],
    'vader_std_w':     PRICE_FEATURES + ['vader_std_wmean_lag1',   'tweet_count_lag1'],
    'vader_crypto':    PRICE_FEATURES + ['vader_crypto_mean_lag1', 'tweet_count_lag1'],
    'vader_crypto_w':  PRICE_FEATURES + ['vader_crypto_wmean_lag1','tweet_count_lag1'],
    'finbert':         PRICE_FEATURES + ['finbert_mean_lag1',      'tweet_count_lag1'],
    'finbert_w':       PRICE_FEATURES + ['finbert_wmean_lag1',     'tweet_count_lag1'],
}

# Walk-forward loop
print(f'\nRunning {N_FOLDS} folds × {len(VARIANTS)} variants = {N_FOLDS * len(VARIANTS)} model fits...\n')

predictions = {v: [] for v in VARIANTS}
actuals = []
dates   = []

import time
start = time.time()

for fold_idx in range(N_FOLDS):
    train_end = MIN_TRAIN_SIZE + fold_idx
    train_fold = df_p.iloc[:train_end].copy()
    test_fold  = df_p.iloc[train_end:train_end + 1].copy()

    actuals.append(test_fold['y'].values[0])
    dates.append(test_fold['ds'].values[0])

    for variant_name, features in VARIANTS.items():
        m = Prophet(yearly_seasonality=False, weekly_seasonality=False,
                    daily_seasonality=False, growth='linear',
                    changepoint_prior_scale=0.05)
        for f in features:
            m.add_regressor(f)
        m.fit(train_fold[['ds', 'y'] + features])
        forecast = m.predict(test_fold[['ds'] + features])
        predictions[variant_name].append(forecast['yhat'].values[0])

    if (fold_idx + 1) % 5 == 0 or fold_idx == N_FOLDS - 1:
        elapsed = time.time() - start
        print(f'  Fold {fold_idx+1}/{N_FOLDS} done  ({elapsed:.0f}s elapsed)')

print(f'\nAll folds complete in {(time.time()-start)/60:.1f} minutes')

# Build results dataframe
results_df = pd.DataFrame({
    'date':   dates,
    'actual': actuals,
    **{v: predictions[v] for v in VARIANTS}
})

# Compute metrics for each variant
from sklearn.metrics import mean_squared_error, mean_absolute_error
print(f'\n{"Variant":<20s}  {"RMSE":>7s}  {"MAE":>7s}  {"DA":>7s}')
print('-' * 50)
metrics = {}
for v in VARIANTS:
    pred = np.array(predictions[v])
    act  = np.array(actuals)
    rmse = mean_squared_error(act, pred) ** 0.5
    mae  = mean_absolute_error(act, pred)
    # Directional accuracy
    pred_change = np.diff(pred)
    act_change  = np.diff(act)
    da = (np.sign(pred_change) == np.sign(act_change)).mean()
    metrics[v] = {'rmse': rmse, 'mae': mae, 'da': da}
    print(f'{v:<20s}  {rmse:>7.4f}  {mae:>7.4f}  {da:>7.2%}')

# Save everything
results_df.to_parquet(f'{PROJECT_ROOT}/data/processed/walkforward_prophet.parquet', index=False)
print('\nSaved walk-forward predictions.')

In [ ]:
from scipy import stats

results_df = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/walkforward_prophet.parquet')
print(f'Folds available: {len(results_df)}')

def diebold_mariano(actual, pred1, pred2, h=1, loss='se'):
    """
    Diebold-Mariano test with Harvey-Leybourne-Newbold small-sample correction.

    actual : 1D array of true values
    pred1  : 1D array of model 1 predictions
    pred2  : 1D array of model 2 predictions
    h      : forecast horizon (1 for one-step-ahead)
    loss   : 'se' = squared error, 'ae' = absolute error

    Returns (DM_statistic, p_value).
    Positive DM = model 1 has HIGHER loss than model 2 (model 2 is better).
    """
    e1 = actual - pred1
    e2 = actual - pred2

    if loss == 'se':
        d = e1**2 - e2**2
    else:  # absolute error
        d = np.abs(e1) - np.abs(e2)

    T = len(d)
    d_mean = d.mean()

    # Variance with Newey-West-style correction for autocorrelation up to lag h-1
    gamma_0 = ((d - d_mean) ** 2).mean()
    gamma_sum = 0
    for k in range(1, h):
        gamma_k = ((d[k:] - d_mean) * (d[:-k] - d_mean)).mean()
        gamma_sum += 2 * gamma_k
    var_d = (gamma_0 + gamma_sum) / T

    if var_d <= 0:
        return np.nan, np.nan

    DM = d_mean / np.sqrt(var_d)

    # Harvey-Leybourne-Newbold small-sample correction
    hln_factor = np.sqrt((T + 1 - 2*h + h*(h-1)/T) / T)
    DM_corrected = DM * hln_factor

    # p-value from t-distribution with T-1 degrees of freedom (HLN recommendation)
    p_value = 2 * (1 - stats.t.cdf(np.abs(DM_corrected), df=T-1))

    return DM_corrected, p_value


actual = results_df['actual'].values
baseline_pred = results_df['baseline'].values

print(f'\n{"Variant":<20s}  {"RMSE":>7s}  {"DM stat":>9s}  {"p-value":>9s}  {"Interpretation":<30s}')
print('-' * 85)

variants = ['vader_std', 'vader_std_w', 'vader_crypto', 'vader_crypto_w', 'finbert', 'finbert_w']

# Baseline RMSE for reference
baseline_rmse = np.sqrt(np.mean((actual - baseline_pred)**2))
print(f'{"baseline":<20s}  {baseline_rmse:>7.4f}  {"—":>9s}  {"—":>9s}  reference')

for v in variants:
    v_pred = results_df[v].values
    v_rmse = np.sqrt(np.mean((actual - v_pred)**2))

    DM, p = diebold_mariano(actual, baseline_pred, v_pred, h=1, loss='se')

    # Interpretation
    if np.isnan(p):
        interp = 'identical errors'
    elif p > 0.10:
        interp = 'no significant difference'
    elif p > 0.05:
        interp = 'marginal (p<0.10)'
    elif p > 0.01:
        interp = '* significant (p<0.05)'
    else:
        interp = '** highly significant (p<0.01)'

    if DM > 0:
        direction = '  sentiment BETTER'
    else:
        direction = '  baseline BETTER'

    print(f'{v:<20s}  {v_rmse:>7.4f}  {DM:>+9.3f}  {p:>9.4f}  {interp}{direction if p < 0.10 else ""}')

print('\nNote: positive DM = sentiment outperforms baseline; negative = baseline outperforms.')

In [ ]:
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler

df = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/weekly_features_final.parquet')
df = df.sort_values('date').reset_index(drop=True)

# Feature definitions (mirrors Prophet variants)
PRICE_FEATURES = ['vol_lag1', 'vol_lag2', 'vol_lag4',
                  'return_lag1', 'abs_return_lag1',
                  'return_std_4w', 'return_mean_4w']

def make_sequences(df, feature_cols, target_col='realised_vol', L=4):
    """
    Turn a dataframe into LSTM-ready sequences.

    For each week i (from L to N), produces:
      X[i] = feature matrix of shape (L, n_features) covering weeks i-L..i-1
      y[i] = target value for week i

    Returns (X, y) as numpy arrays.
    """
    X_list, y_list = [], []
    for i in range(L, len(df)):
        X_list.append(df.iloc[i-L:i][feature_cols].values)
        y_list.append(df.iloc[i][target_col])
    return np.array(X_list), np.array(y_list)

# Quick test with L=4
L = 4
X, y = make_sequences(df, PRICE_FEATURES, L=L)
print(f'Sequence length L           : {L}')
print(f'X shape (samples, L, feats) : {X.shape}')
print(f'y shape                     : {y.shape}')
print(f'First sequence first row    : {X[0][0]}')
print(f'First target                : {y[0]:.4f}')

In [ ]:
class VolatilityLSTM(nn.Module):
    def __init__(self, n_features, hidden_size=16, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            dropout=0.0,  # only relevant for num_layers > 1
        )
        self.dropout = nn.Dropout(dropout)
        self.head = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x: (batch, L, n_features)
        out, (h_n, _) = self.lstm(x)
        # Use the last timestep's hidden state
        h = out[:, -1, :]
        h = self.dropout(h)
        return self.head(h).squeeze(-1)

# Quick instantiation test
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device available: {device}')

test_model = VolatilityLSTM(n_features=len(PRICE_FEATURES)).to(device)
n_params = sum(p.numel() for p in test_model.parameters())
print(f'\nModel: VolatilityLSTM(n_features={len(PRICE_FEATURES)}, hidden_size=16)')
print(f'Total parameters: {n_params:,}')

# Verify forward pass works
test_X = torch.randn(8, L, len(PRICE_FEATURES)).to(device)
test_out = test_model(test_X)
print(f'Test input shape : {tuple(test_X.shape)}')
print(f'Test output shape: {tuple(test_out.shape)}')
print(f'Test output sample: {test_out[:3].detach().cpu().numpy()}')

In [ ]:
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Reuse the sequences we built earlier (price features, L=4)
print(f'X shape: {X.shape}, y shape: {y.shape}')

# Same temporal split as Prophet baseline:
# Total weeks after lagging = 53, test = 8.
# With L=4 dropped for sequence warmup, samples = 49.
# So our test sequences are the last 8 of these 49 samples.
n_test = 8
X_train_raw, y_train = X[:-n_test], y[:-n_test]
X_test_raw,  y_test  = X[-n_test:], y[-n_test:]

# We further hold out the LAST 4 training samples as validation for early stopping
n_val = 4
X_val_raw, y_val = X_train_raw[-n_val:], y_train[-n_val:]
X_train_raw, y_train = X_train_raw[:-n_val], y_train[:-n_val]

print(f'Train samples: {len(y_train)}')
print(f'Val samples  : {len(y_val)}')
print(f'Test samples : {len(y_test)}')

# Scale features — fit on train only, apply to all
# Flatten time dimension for scaling, then reshape back
n_features = X.shape[2]
scaler_X = StandardScaler()
X_train_flat = X_train_raw.reshape(-1, n_features)
scaler_X.fit(X_train_flat)

def scale(X_raw):
    flat = X_raw.reshape(-1, n_features)
    return scaler_X.transform(flat).reshape(X_raw.shape)

X_train = scale(X_train_raw)
X_val   = scale(X_val_raw)
X_test  = scale(X_test_raw)

# Convert to tensors
X_train_t = torch.FloatTensor(X_train).to(device)
y_train_t = torch.FloatTensor(y_train).to(device)
X_val_t   = torch.FloatTensor(X_val).to(device)
y_val_t   = torch.FloatTensor(y_val).to(device)
X_test_t  = torch.FloatTensor(X_test).to(device)
y_test_t  = torch.FloatTensor(y_test).to(device)

# Model + optimiser
model = VolatilityLSTM(n_features=n_features, hidden_size=16, dropout=0.2).to(device)
optimizer = optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()

# Training loop with early stopping
MAX_EPOCHS = 200
PATIENCE   = 20
best_val_loss = float('inf')
patience_counter = 0
best_state = None

train_losses, val_losses = [], []

for epoch in range(MAX_EPOCHS):
    model.train()
    optimizer.zero_grad()
    pred = model(X_train_t)
    loss = criterion(pred, y_train_t)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_t)
        val_loss = criterion(val_pred, y_val_t).item()

    train_losses.append(loss.item())
    val_losses.append(val_loss)

    if val_loss < best_val_loss - 1e-5:
        best_val_loss = val_loss
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping at epoch {epoch+1} (best val MSE: {best_val_loss:.4f})')
            break

# Restore best weights
model.load_state_dict(best_state)

# Test set evaluation
model.eval()
with torch.no_grad():
    test_pred = model(X_test_t).cpu().numpy()

# Metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error
rmse = mean_squared_error(y_test, test_pred) ** 0.5
mae  = mean_absolute_error(y_test, test_pred)
pred_change = np.diff(test_pred)
actual_change = np.diff(y_test)
da = (np.sign(pred_change) == np.sign(actual_change)).mean()

print(f'\n--- LSTM Baseline (price-only) Test Results ---')
print(f'  RMSE                : {rmse:.4f}')
print(f'  MAE                 : {mae:.4f}')
print(f'  Directional accuracy: {da:.2%}')
print(f'\nTest predictions:')
for i in range(len(y_test)):
    print(f'  actual={y_test[i]:.3f}  predicted={test_pred[i]:.3f}  '
          f'error={y_test[i]-test_pred[i]:+.3f}')

print(f'\nFor comparison — Prophet baseline single-window: RMSE 0.1826, DA 71.43%')

In [ ]:
import time
import warnings
warnings.filterwarnings('ignore')

df = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/weekly_features_final.parquet')
df = df.sort_values('date').reset_index(drop=True)

# Define feature sets — mirrors Prophet
PRICE_FEATURES = ['vol_lag1', 'vol_lag2', 'vol_lag4',
                  'return_lag1', 'abs_return_lag1',
                  'return_std_4w', 'return_mean_4w']

VARIANTS = {
    'baseline':        PRICE_FEATURES,
    'vader_std':       PRICE_FEATURES + ['vader_std_mean_lag1',     'tweet_count_lag1'],
    'vader_std_w':     PRICE_FEATURES + ['vader_std_wmean_lag1',    'tweet_count_lag1'],
    'vader_crypto':    PRICE_FEATURES + ['vader_crypto_mean_lag1',  'tweet_count_lag1'],
    'vader_crypto_w':  PRICE_FEATURES + ['vader_crypto_wmean_lag1', 'tweet_count_lag1'],
    'finbert':         PRICE_FEATURES + ['finbert_mean_lag1',       'tweet_count_lag1'],
    'finbert_w':       PRICE_FEATURES + ['finbert_wmean_lag1',      'tweet_count_lag1'],
}

L = 4
MIN_TRAIN_SIZE = 25
N_VAL_PER_FOLD = 4

def train_lstm_and_predict(X_train_seqs, y_train_seq,
                           X_val_seqs,   y_val_seq,
                           X_test_seq,
                           n_features, seed=42):
    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    Xt  = torch.FloatTensor(X_train_seqs).to(device)
    yt  = torch.FloatTensor(y_train_seq).to(device)
    Xv  = torch.FloatTensor(X_val_seqs).to(device)
    yv  = torch.FloatTensor(y_val_seq).to(device)
    Xts = torch.FloatTensor(X_test_seq).to(device)

    model = VolatilityLSTM(n_features=n_features, hidden_size=16, dropout=0.2).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    criterion = nn.MSELoss()

    best_val = float('inf')
    best_state = None
    patience_counter = 0

    for epoch in range(200):
        model.train()
        optimizer.zero_grad()
        loss = criterion(model(Xt), yt)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_loss = criterion(model(Xv), yv).item()

        if val_loss < best_val - 1e-5:
            best_val = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= 20:
                break

    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        pred = model(Xts).cpu().numpy()
    return pred[0]


# Pre-build sequences for each variant
print('Building sequences for each variant...')
variant_data = {}
for name, feats in VARIANTS.items():
    X_full, y_full = make_sequences(df, feats, L=L)
    variant_data[name] = {'X': X_full, 'y': y_full, 'features': feats}
    print(f'  {name:20s} → X {X_full.shape}, y {y_full.shape}')

# Walk-forward configuration. We forecast each week from index MIN_TRAIN_SIZE upward.
# Indices below refer to the SEQUENCE array, where index i predicts df.iloc[i+L]'s target.
total_sequences = variant_data['baseline']['X'].shape[0]
forecast_indices = list(range(MIN_TRAIN_SIZE - L, total_sequences))
n_folds = len(forecast_indices)
print(f'\nWalk-forward folds: {n_folds}')
print(f'Will fit {n_folds * len(VARIANTS)} models total\n')

predictions = {v: [] for v in VARIANTS}
actuals = []
dates = []
start = time.time()

for fold_pos, i in enumerate(forecast_indices):
    actuals.append(variant_data['baseline']['y'][i])
    dates.append(df.iloc[i + L]['date'])

    for variant_name in VARIANTS:
        d = variant_data[variant_name]
        X_full, y_full = d['X'], d['y']
        n_features = X_full.shape[2]

        # Training set: sequences 0..i-1
        X_train_all = X_full[:i]
        y_train_all = y_full[:i]

        # Split last N_VAL_PER_FOLD as validation
        X_train_raw = X_train_all[:-N_VAL_PER_FOLD]
        y_train     = y_train_all[:-N_VAL_PER_FOLD]
        X_val_raw   = X_train_all[-N_VAL_PER_FOLD:]
        y_val       = y_train_all[-N_VAL_PER_FOLD:]

        # Test: single sequence at index i
        X_test_raw = X_full[i:i+1]

        # Scale (fit on this fold's training data only)
        sc = StandardScaler()
        sc.fit(X_train_raw.reshape(-1, n_features))
        def _scale(arr):
            return sc.transform(arr.reshape(-1, n_features)).reshape(arr.shape)
        X_train = _scale(X_train_raw)
        X_val   = _scale(X_val_raw)
        X_test  = _scale(X_test_raw)

        pred = train_lstm_and_predict(
            X_train, y_train, X_val, y_val, X_test, n_features=n_features, seed=42
        )
        predictions[variant_name].append(pred)

    if (fold_pos + 1) % 5 == 0 or fold_pos == n_folds - 1:
        elapsed = time.time() - start
        print(f'  Fold {fold_pos+1}/{n_folds} done  ({elapsed/60:.1f} min elapsed)')

print(f'\nAll folds complete in {(time.time()-start)/60:.1f} minutes')

# Results table
print(f'\n{"Variant":<20s}  {"RMSE":>7s}  {"MAE":>7s}  {"DA":>7s}')
print('-' * 50)
for v in VARIANTS:
    pred = np.array(predictions[v])
    act = np.array(actuals)
    rmse = mean_squared_error(act, pred) ** 0.5
    mae  = mean_absolute_error(act, pred)
    da   = (np.sign(np.diff(pred)) == np.sign(np.diff(act))).mean()
    print(f'{v:<20s}  {rmse:>7.4f}  {mae:>7.4f}  {da:>7.2%}')

# Save
results_df_lstm = pd.DataFrame({
    'date': dates,
    'actual': actuals,
    **{v: predictions[v] for v in VARIANTS}
})
results_df_lstm.to_parquet(f'{PROJECT_ROOT}/data/processed/walkforward_lstm.parquet', index=False)
print('\nSaved walk-forward LSTM predictions.')

In [ ]:
results_df_lstm = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/walkforward_lstm.parquet')
print(f'LSTM folds available: {len(results_df_lstm)}')

actual = results_df_lstm['actual'].values
baseline_pred = results_df_lstm['baseline'].values

print(f'\n{"Variant":<20s}  {"RMSE":>7s}  {"DM stat":>9s}  {"p-value":>9s}  {"Interpretation":<35s}')
print('-' * 90)

baseline_rmse = np.sqrt(np.mean((actual - baseline_pred)**2))
print(f'{"baseline":<20s}  {baseline_rmse:>7.4f}  {"—":>9s}  {"—":>9s}  reference')

variants = ['vader_std', 'vader_std_w', 'vader_crypto', 'vader_crypto_w', 'finbert', 'finbert_w']

for v in variants:
    v_pred = results_df_lstm[v].values
    v_rmse = np.sqrt(np.mean((actual - v_pred)**2))

    DM, p = diebold_mariano(actual, baseline_pred, v_pred, h=1, loss='se')

    if np.isnan(p):
        interp = 'identical errors'
    elif p > 0.10:
        interp = 'no significant difference'
    elif p > 0.05:
        interp = 'marginal (p<0.10)'
    elif p > 0.01:
        interp = '* significant (p<0.05)'
    else:
        interp = '** highly significant (p<0.01)'

    direction = ''
    if not np.isnan(p) and p < 0.10:
        direction = '  sentiment BETTER' if DM > 0 else '  baseline BETTER'

    print(f'{v:<20s}  {v_rmse:>7.4f}  {DM:>+9.3f}  {p:>9.4f}  {interp}{direction}')

print('\nNote: positive DM = sentiment outperforms baseline; negative = baseline outperforms.')

In [ ]:
# Load both Prophet and LSTM walk-forward results
prophet_df = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/walkforward_prophet.parquet')
lstm_df    = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/walkforward_lstm.parquet')

print(f'Prophet folds: {len(prophet_df)}')
print(f'LSTM folds   : {len(lstm_df)}')

# Define regime by tercile of actual realised volatility
actual = prophet_df['actual'].values
low_thresh  = np.quantile(actual, 1/3)
high_thresh = np.quantile(actual, 2/3)

regime = np.where(actual <= low_thresh, 'low',
          np.where(actual >= high_thresh, 'high', 'medium'))

prophet_df['regime'] = regime
lstm_df['regime']    = regime

print(f'\nRegime thresholds:')
print(f'  Low-vol  : actual ≤ {low_thresh:.3f}')
print(f'  Medium   : {low_thresh:.3f} < actual < {high_thresh:.3f}')
print(f'  High-vol : actual ≥ {high_thresh:.3f}')

print(f'\nRegime counts:')
for r in ['low', 'medium', 'high']:
    n = (regime == r).sum()
    print(f'  {r:7s}: {n} folds')

# Compute per-regime RMSE for each variant in each model family
variants = ['baseline', 'vader_std', 'vader_std_w', 'vader_crypto',
            'vader_crypto_w', 'finbert', 'finbert_w']

def per_regime_rmse(df, variants):
    rows = []
    for r in ['low', 'medium', 'high']:
        sub = df[df['regime'] == r]
        row = {'regime': r, 'n': len(sub)}
        for v in variants:
            err = sub['actual'] - sub[v]
            row[v] = np.sqrt((err ** 2).mean())
        rows.append(row)
    return pd.DataFrame(rows)

prophet_regime = per_regime_rmse(prophet_df, variants)
lstm_regime    = per_regime_rmse(lstm_df, variants)

print('\n' + '=' * 80)
print('PROPHET: Per-regime RMSE')
print('=' * 80)
print(prophet_regime.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

print('\n' + '=' * 80)
print('LSTM: Per-regime RMSE')
print('=' * 80)
print(lstm_regime.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# Show the % improvement of each sentiment variant vs baseline, per regime
def improvement_pct(df_regime, variants):
    rows = []
    for r in ['low', 'medium', 'high']:
        row = df_regime[df_regime['regime'] == r].iloc[0]
        base = row['baseline']
        improvements = {'regime': r}
        for v in variants:
            if v == 'baseline':
                continue
            delta = (row[v] - base) / base * 100
            improvements[v] = delta
        rows.append(improvements)
    return pd.DataFrame(rows)

print('\n' + '=' * 80)
print('PROPHET: % change in RMSE vs baseline (negative = sentiment better)')
print('=' * 80)
print(improvement_pct(prophet_regime, variants).to_string(index=False, float_format=lambda x: f'{x:+.2f}%'))

print('\n' + '=' * 80)
print('LSTM: % change in RMSE vs baseline (negative = sentiment better)')
print('=' * 80)
print(improvement_pct(lstm_regime, variants).to_string(index=False, float_format=lambda x: f'{x:+.2f}%'))

# Save
prophet_regime.to_parquet(f'{PROJECT_ROOT}/data/processed/regime_prophet.parquet', index=False)
lstm_regime.to_parquet(f'{PROJECT_ROOT}/data/processed/regime_lstm.parquet', index=False)
print('\nSaved regime breakdowns.')

In [ ]:
import time

# Reload base data
df = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/weekly_features_final.parquet')
df = df.sort_values('date').reset_index(drop=True)

# Build additional lag features on the unfiltered sentiment series
# (we need lag-2 and lag-3 versions of the columns we used)
for col in ['vader_std_wmean', 'vader_crypto_wmean', 'finbert_wmean']:
    df[f'{col}_lag2'] = df[col].shift(2)
    df[f'{col}_lag3'] = df[col].shift(3)

# tweet_count lags
df['tweet_count_lag2'] = df['tweet_count'].shift(2)
df['tweet_count_lag3'] = df['tweet_count'].shift(3)

# Drop any new NaN rows from the deeper lags
df = df.dropna().reset_index(drop=True)
print(f'Rows after extended lagging : {len(df)}')

# Define the three lag variants for VADER std weighted (the best DA performer)
PRICE_FEATURES = ['vol_lag1', 'vol_lag2', 'vol_lag4',
                  'return_lag1', 'abs_return_lag1',
                  'return_std_4w', 'return_mean_4w']

LAG_VARIANTS = {
    'lag1': PRICE_FEATURES + ['vader_std_wmean_lag1', 'tweet_count_lag1'],
    'lag2': PRICE_FEATURES + ['vader_std_wmean_lag2', 'tweet_count_lag2'],
    'lag3': PRICE_FEATURES + ['vader_std_wmean_lag3', 'tweet_count_lag3'],
}

# Build sequences for each lag variant
L = 4
lag_data = {}
for name, feats in LAG_VARIANTS.items():
    X_full, y_full = make_sequences(df, feats, L=L)
    lag_data[name] = {'X': X_full, 'y': y_full}
    print(f'  {name}: X {X_full.shape}, y {y_full.shape}')

# Walk-forward — same config as before
MIN_TRAIN_SIZE = 25
N_VAL_PER_FOLD = 4

total_sequences = lag_data['lag1']['X'].shape[0]
forecast_indices = list(range(MIN_TRAIN_SIZE - L, total_sequences))
n_folds = len(forecast_indices)
print(f'\nWalk-forward folds: {n_folds}')
print(f'Will fit {n_folds * len(LAG_VARIANTS)} models total\n')

predictions = {v: [] for v in LAG_VARIANTS}
actuals = []
start = time.time()

for fold_pos, i in enumerate(forecast_indices):
    actuals.append(lag_data['lag1']['y'][i])

    for variant_name in LAG_VARIANTS:
        d = lag_data[variant_name]
        X_full, y_full = d['X'], d['y']
        n_features = X_full.shape[2]

        X_train_all = X_full[:i]
        y_train_all = y_full[:i]
        X_train_raw = X_train_all[:-N_VAL_PER_FOLD]
        y_train     = y_train_all[:-N_VAL_PER_FOLD]
        X_val_raw   = X_train_all[-N_VAL_PER_FOLD:]
        y_val       = y_train_all[-N_VAL_PER_FOLD:]
        X_test_raw  = X_full[i:i+1]

        sc = StandardScaler()
        sc.fit(X_train_raw.reshape(-1, n_features))
        def _scale(arr):
            return sc.transform(arr.reshape(-1, n_features)).reshape(arr.shape)
        X_train = _scale(X_train_raw)
        X_val   = _scale(X_val_raw)
        X_test  = _scale(X_test_raw)

        pred = train_lstm_and_predict(
            X_train, y_train, X_val, y_val, X_test, n_features=n_features, seed=42
        )
        predictions[variant_name].append(pred)

    if (fold_pos + 1) % 5 == 0 or fold_pos == n_folds - 1:
        print(f'  Fold {fold_pos+1}/{n_folds} done  ({(time.time()-start)/60:.1f} min)')

print(f'\nAll folds complete in {(time.time()-start)/60:.1f} minutes')

# Compute metrics
print(f'\n{"Lag":<6s}  {"RMSE":>7s}  {"MAE":>7s}  {"DA":>7s}')
print('-' * 35)
act = np.array(actuals)
for v in LAG_VARIANTS:
    pred = np.array(predictions[v])
    rmse = np.sqrt(np.mean((act - pred) ** 2))
    mae  = np.mean(np.abs(act - pred))
    da   = (np.sign(np.diff(pred)) == np.sign(np.diff(act))).mean()
    print(f'{v:<6s}  {rmse:>7.4f}  {mae:>7.4f}  {da:>7.2%}')

print(f'\nFor comparison — baseline LSTM (no sentiment): RMSE 0.2164, DA 48.15%')

# Save
results_lag = pd.DataFrame({
    'actual': actuals,
    **{v: predictions[v] for v in LAG_VARIANTS}
})
results_lag.to_parquet(f'{PROJECT_ROOT}/data/processed/lag_sensitivity_lstm.parquet', index=False)

In [ ]:
import time

# Reload base data and re-build lag-3 features
df = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/weekly_features_final.parquet')
df = df.sort_values('date').reset_index(drop=True)

# Build lag-3 versions of all sentiment columns
for col in ['vader_std_mean', 'vader_crypto_mean', 'finbert_mean',
            'vader_std_wmean', 'vader_crypto_wmean', 'finbert_wmean']:
    df[f'{col}_lag3'] = df[col].shift(3)

df['tweet_count_lag3'] = df['tweet_count'].shift(3)

df = df.dropna().reset_index(drop=True)
print(f'Rows after lag-3 prep: {len(df)}')

PRICE_FEATURES = ['vol_lag1', 'vol_lag2', 'vol_lag4',
                  'return_lag1', 'abs_return_lag1',
                  'return_std_4w', 'return_mean_4w']

VARIANTS_LAG3 = {
    'baseline':        PRICE_FEATURES,
    'vader_std':       PRICE_FEATURES + ['vader_std_mean_lag3',     'tweet_count_lag3'],
    'vader_std_w':     PRICE_FEATURES + ['vader_std_wmean_lag3',    'tweet_count_lag3'],
    'vader_crypto':    PRICE_FEATURES + ['vader_crypto_mean_lag3',  'tweet_count_lag3'],
    'vader_crypto_w':  PRICE_FEATURES + ['vader_crypto_wmean_lag3', 'tweet_count_lag3'],
    'finbert':         PRICE_FEATURES + ['finbert_mean_lag3',       'tweet_count_lag3'],
    'finbert_w':       PRICE_FEATURES + ['finbert_wmean_lag3',      'tweet_count_lag3'],
}

L = 4
variant_data = {}
for name, feats in VARIANTS_LAG3.items():
    X_full, y_full = make_sequences(df, feats, L=L)
    variant_data[name] = {'X': X_full, 'y': y_full}

print(f'Sequences per variant: {variant_data["baseline"]["X"].shape}\n')

MIN_TRAIN_SIZE = 25
N_VAL_PER_FOLD = 4
total_sequences = variant_data['baseline']['X'].shape[0]
forecast_indices = list(range(MIN_TRAIN_SIZE - L, total_sequences))
n_folds = len(forecast_indices)
print(f'Walk-forward folds: {n_folds}')
print(f'Will fit {n_folds * len(VARIANTS_LAG3)} models total\n')

predictions = {v: [] for v in VARIANTS_LAG3}
actuals = []
dates = []
start = time.time()

for fold_pos, i in enumerate(forecast_indices):
    actuals.append(variant_data['baseline']['y'][i])
    dates.append(df.iloc[i + L]['date'])

    for variant_name in VARIANTS_LAG3:
        d = variant_data[variant_name]
        X_full, y_full = d['X'], d['y']
        n_features = X_full.shape[2]

        X_train_all = X_full[:i]
        y_train_all = y_full[:i]
        X_train_raw = X_train_all[:-N_VAL_PER_FOLD]
        y_train     = y_train_all[:-N_VAL_PER_FOLD]
        X_val_raw   = X_train_all[-N_VAL_PER_FOLD:]
        y_val       = y_train_all[-N_VAL_PER_FOLD:]
        X_test_raw  = X_full[i:i+1]

        sc = StandardScaler()
        sc.fit(X_train_raw.reshape(-1, n_features))
        def _scale(arr):
            return sc.transform(arr.reshape(-1, n_features)).reshape(arr.shape)
        X_train = _scale(X_train_raw)
        X_val   = _scale(X_val_raw)
        X_test  = _scale(X_test_raw)

        pred = train_lstm_and_predict(
            X_train, y_train, X_val, y_val, X_test, n_features=n_features, seed=42
        )
        predictions[variant_name].append(pred)

    if (fold_pos + 1) % 5 == 0 or fold_pos == n_folds - 1:
        print(f'  Fold {fold_pos+1}/{n_folds} done  ({(time.time()-start)/60:.1f} min)')

print(f'\nAll folds complete in {(time.time()-start)/60:.1f} minutes')

# Results table
print(f'\n{"Variant":<20s}  {"RMSE":>7s}  {"MAE":>7s}  {"DA":>7s}')
print('-' * 50)
act = np.array(actuals)
for v in VARIANTS_LAG3:
    pred = np.array(predictions[v])
    rmse = np.sqrt(np.mean((act - pred) ** 2))
    mae  = np.mean(np.abs(act - pred))
    da   = (np.sign(np.diff(pred)) == np.sign(np.diff(act))).mean()
    print(f'{v:<20s}  {rmse:>7.4f}  {mae:>7.4f}  {da:>7.2%}')

# DM tests vs baseline
print(f'\n{"Variant":<20s}  {"DM stat":>9s}  {"p-value":>9s}  {"Interpretation":<35s}')
print('-' * 85)
baseline_pred = np.array(predictions['baseline'])
for v in VARIANTS_LAG3:
    if v == 'baseline':
        continue
    v_pred = np.array(predictions[v])
    DM, p = diebold_mariano(act, baseline_pred, v_pred, h=1, loss='se')
    if np.isnan(p):
        interp = 'identical errors'
    elif p > 0.10:
        interp = 'no significant difference'
    elif p > 0.05:
        interp = 'marginal (p<0.10)'
    elif p > 0.01:
        interp = '* significant (p<0.05)'
    else:
        interp = '** highly significant (p<0.01)'
    direction = ''
    if not np.isnan(p) and p < 0.10:
        direction = '  sentiment BETTER' if DM > 0 else '  baseline BETTER'
    print(f'{v:<20s}  {DM:>+9.3f}  {p:>9.4f}  {interp}{direction}')

# Save
results_df_lag3 = pd.DataFrame({
    'date': dates,
    'actual': actuals,
    **{v: predictions[v] for v in VARIANTS_LAG3}
})
results_df_lag3.to_parquet(f'{PROJECT_ROOT}/data/processed/walkforward_lstm_lag3.parquet', index=False)
print('\nSaved lag-3 walk-forward results.')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib as mpl
import os

# Figures directory
FIGDIR = f'{PROJECT_ROOT}/results/figures'
os.makedirs(FIGDIR, exist_ok=True)

# Global publication style
mpl.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'figure.dpi': 110,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'grid.linestyle': '--',
    'lines.linewidth': 1.5,
})

# Consistent color palette across all figures
COLOR_PRICE      = '#F7931A'   # Bitcoin orange
COLOR_VOL        = '#C0392B'   # crimson
COLOR_BASELINE   = '#444444'   # dark grey for baseline
COLOR_VADER      = '#2E86AB'   # blue
COLOR_VADER_CRY  = '#A23B72'   # purple
COLOR_FINBERT    = '#3D9970'   # green
COLOR_HIGHLIGHT  = '#E74C3C'   # red for "best" / annotation

# ---- Figure 4.1: BTC price + realised volatility ----
btc = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/btc_with_volatility.parquet')
btc['date'] = pd.to_datetime(btc['date'])

fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

ax[0].plot(btc['date'], btc['close'], color=COLOR_PRICE)
ax[0].set_ylabel('BTC/USDT price (USD)')
ax[0].set_title('Bitcoin price and realised volatility, Feb 2021 – Jan 2023')
ax[0].yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# Annotate key regime events
events = [
    ('2021-05-19', 'China mining\nban'),
    ('2021-11-10', 'All-time high\n$69K'),
    ('2022-05-09', 'Terra/Luna\ncollapse'),
    ('2022-11-09', 'FTX\ncollapse'),
]
for d, label in events:
    d = pd.Timestamp(d)
    if d >= btc['date'].min() and d <= btc['date'].max():
        ax[0].axvline(d, color='grey', linestyle=':', alpha=0.4, linewidth=0.8)
        ax[0].annotate(label, xy=(d, btc[btc['date'].dt.date == d.date()]['close'].values[0]
                                  if (btc['date'].dt.date == d.date()).any() else 30000),
                       xytext=(5, -25), textcoords='offset points',
                       fontsize=7.5, color='#555', alpha=0.9)

ax[1].plot(btc['date'], btc['realised_vol'], color=COLOR_VOL)
ax[1].set_ylabel('Realised volatility\n(30-day, annualised)')
ax[1].set_xlabel('Date')
for d, _ in events:
    d = pd.Timestamp(d)
    if d >= btc['date'].min() and d <= btc['date'].max():
        ax[1].axvline(d, color='grey', linestyle=':', alpha=0.4, linewidth=0.8)

plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig_4_1_btc_price_volatility.png')
plt.show()
print(f'Saved: fig_4_1_btc_price_volatility.png')

In [ ]:
weekly_features = pd.read_parquet(f'{PROJECT_ROOT}/data/processed/weekly_features_final.parquet')
weekly_features['date'] = pd.to_datetime(weekly_features['date'])

fig, ax = plt.subplots(2, 1, figsize=(8, 5), sharex=True)

# Top panel: tweet volume per week
ax[0].fill_between(weekly_features['date'], 0, weekly_features['tweet_count'],
                   color=COLOR_PRICE, alpha=0.35, linewidth=0)
ax[0].plot(weekly_features['date'], weekly_features['tweet_count'],
           color=COLOR_PRICE, linewidth=1.2)
ax[0].set_ylabel('Tweets per week\n(after bot filtering)')
ax[0].set_title('Weekly tweet volume and follower-weighted sentiment')
ax[0].yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x, _: f'{x/1000:.0f}K'))

# Bottom panel: three sentiment methods (weighted)
ax[1].plot(weekly_features['date'], weekly_features['vader_std_wmean'],
           color=COLOR_VADER, label='VADER (standard)', alpha=0.9)
ax[1].plot(weekly_features['date'], weekly_features['vader_crypto_wmean'],
           color=COLOR_VADER_CRY, label='VADER (crypto-augmented)', alpha=0.9)
ax[1].plot(weekly_features['date'], weekly_features['finbert_wmean'],
           color=COLOR_FINBERT, label='FinBERT', alpha=0.9)
ax[1].axhline(0, color='grey', linestyle='-', alpha=0.4, linewidth=0.7)
ax[1].set_ylabel('Sentiment\n(follower-weighted mean)')
ax[1].set_xlabel('Date')
ax[1].legend(loc='lower left', frameon=True, framealpha=0.9, ncol=3)

plt.tight_layout()
plt.savefig(f'{FIGDIR}/fig_4_2_tweet_volume_sentiment.png')
plt.show()
print('Saved: fig_4_2_tweet_volume_sentiment.png')